In [ ]:
import os
import json
import pandas as pd
import time

In [ ]:
scenario = 0
path = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "db-results", f"resultsEurostar{scenario}")

In [ ]:
stations = {
    "Gv": "The Hague",
    "Dt": "Delft",
    "Dtcp": "Delft",
    "Laa": "The Hague",
    "Rtd": "Rotterdam",
    "Shl": "Schiphol",
    "Hfd": "Hoofddorp",
    "Ledn": "Leiden",
    "Sdm": "Schiedam",
    "Rmoa": "Rotterdam",
    "Gvm": "The Hague"
}


In [ ]:
num_trains = {}
scen_time = {}
for scen in [1,2,3,4]:
	with open(os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "data", "railways", "case_study_scenarios", f"2025-07-08_{scen}.json"), "r") as f:
		jsonobj = json.load(f)
		num_trains[scen] = len(jsonobj["trains"])
		scen_time[scen] = max([x["movements"]["endTime"] for x in jsonobj["trains"]]) / 60

### Tipping points

In [ ]:
scenario_tipping_point = 1
lines = [r"\begin{table}", r"\caption{Tipping points found by FlexSIPP for Scenario~" + str(scenario_tipping_point) + r"(" + f"{num_trains[scenario_tipping_point]}" + r" trains over a " + f"{scen_time[scenario_tipping_point]:.0f}" + r"-minute timespan). Times in \texttt{mm:ss}.}", r"\label{tab:tip}", r"\begin{tabular}{llll}", r"\toprule", r"Tipping Point & Train & Location & Delay", r"\midrule"]
with open(os.path.join(path, f"tipping_points_eurostar-{scenario_tipping_point-1}.json"), "r") as f:
    tipping_points = json.load(f)
    for point in tipping_points:
        print(f"Eurostar should reach {point['location']['loc']} before {point['time']:.2f} by delaying trains:")
        print(' and'.join([train + " at " + " and at ".join([f"{node} for {x:.2f}" for node, x in delay.items()]) for train, delay in point["delays"].items() if delay]))
        for train, delay in point["delays"].items():
            for node, d in delay.items():
                lines.append(str(time.strftime('%M:%S', time.gmtime(point["time"]))) + r" & " + str(train) + r" & " + stations[node.split(" ")[-1].split("|")[0]] + r" & " + str(time.strftime('%M:%S', time.gmtime(d))))
        lines.append(r"\midrule")
lines.append(r"\bottomrule")
print()
print("\n".join(lines))

### Timing graph

In [ ]:
df = pd.DataFrame(columns=["Route Creation Network", "Conflict Generation", "Interval Generation", "Flexibility Generation", "Search Time FlexSIPP", "Search Time @MAEDeR", "Found Paths FlexSIPP", "Found Paths @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR"])
df

## Read Data

In [ ]:
for scen in [0,1,2,3]:
    curdir = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "db-results", f"resultsEurostar{scen}")
    if not os.path.isdir(curdir):
        continue
    new_row = {"Route Creation Network": -1, "Conflict Generation": -1, "Interval Generation": -1, "Flexibility Generation": -1, "Search Time FlexSIPP": -1, "Search Time @MAEDeR": -1, "Found Paths FlexSIPP": -1, "Found Paths @MAEDeR": -1}
    # Route creation = block graph + track graph
    with open(os.path.join(curdir, "timing", "TrackGraph.__init__.csv"), "r") as f:
        new_row["Route Creation Network"] = float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "BlockGraph.__init__.csv"), "r") as f:
        new_row["Route Creation Network"] += float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "Scenario.process_blocking_time_intervals.csv"), "r") as f:
        new_row["Conflict Generation"] = float(f.read().split("\n")[1])
    # with open(os.path.join(curdir, "timing", "FSIPP.__init__.csv"), "r") as f:
    #     new_row["Interval Generation"] = float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "Scenario.compute_flexibility.csv"), "r") as f:
        new_row["Flexibility Generation"] = float(f.read().split("\n")[1])
    for alg in ["FlexSIPP", "@MAEDeR"]:
        with open(os.path.join(path, f"fsipp_{alg}_Eurostar_search-{scen}.json"), "r") as f:
            results_data = json.load(f)
            path_lengths = {}
            for item in results_data["Result"]["payloads"]:
                atf = str(item["edge_atf"]["atf"])
                if atf not in path_lengths:
                    path_lengths[atf] = len([x for x in item["payload"] if "state" in x])
            new_row[f"Found Paths {alg}"] = len(path_lengths)
            new_row[f"Search Time {alg}"] = int(results_data["MetaData"]["Search Time"]) / 1000
            new_row[f"Nodes Expanded {alg}"] = int(results_data["MetaData"]["Nodes expanded"]) / 1000
    df.loc[scen] = new_row
df

In [ ]:
time_table = df.drop(columns=["Found Paths FlexSIPP", "Found Paths @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR"])
df_t = time_table.T
df_t.columns = [f"S{i+1}" for i in range(df_t.shape[1])]
latex = df_t.to_latex(caption="Table reporting the components of the runtime in seconds of the two algorithms for four different scenarios.", label="tab:time")
latex = latex.replace("Route Creation Network", r"Creation Routes~$\network$").replace("Generation", "gen.").replace("Time ", "")
lines = latex.split("\n")
for i, line in enumerate(lines):
    if "S1" in lines[i]:
        lines[i] = "Time Components" + lines[i]
print("\n".join(lines))

In [ ]:
path_table = df.drop(columns=["Route Creation Network", "Conflict Generation", "Interval Generation", "Flexibility Generation", "Search Time FlexSIPP", "Search Time @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR"])
latex = path_table.to_latex(caption="Number of paths found to accommodate Eurostar.", label="tab:compare")
lines = latex.split("\n")
for i, line in enumerate(lines):
    if "Found Paths" in lines[i]:
        lines[i] = line.replace("Found Paths", "")
    elif "&" in line:
        lines[i] = str(int(lines[i].split(r" & ")[0]) + 1) + lines[i][1:]
print("\n".join(lines))
path_table